In [1]:
print("faris")

faris


In [2]:
# za svaki bat i:                     # prolazimo kroz svakog šišmiša pojedinačno
#                                     # i = indeks jednog šišmiša u populaciji
 
#     beta = random(0, 1)             # slučajan broj između 0 i 1
#                                     # koristi se da frekvencija bude malo drugačija za svakog šišmiša
 
#     f[i] = f_min + (f_max - f_min) * beta
#                                     # f[i] = frekvencija i-tog šišmiša
#                                     # frekvencija određuje koliko jako mijenja svoje kretanje
#                                     # nije "pozicija", nego parametar koji utiče na pomak
 
#     v[i] = v[i] + (x[i] - best) * f[i]
#                                     # v[i] = brzina i-tog šišmiša
#                                     # brzina govori koliko i u kojem smjeru će se pomjeriti
#                                     # x[i] = trenutna pozicija tog šišmiša
#                                     # best = trenutno najbolje rješenje od svih šišmiša
#                                     # ovom formulom šišmiš koriguje svoju brzinu u odnosu na best
 
#     x_new = x[i] + v[i]
#                                     # x_new = nova kandidatska pozicija za tog jednog šišmiša
#                                     # znači: uzmemo staru poziciju i dodamo brzinu
#                                     # DA — ovo je nova pozicija JEDNOG šišmiša, ovog i-tog
 
#     ako random(0,1) > pulse_rate[i]:
#         x_new = best + epsilon * average_loudness
#                                     # ponekad šišmiš ne ide običnim pomakom
#                                     # nego skoči blizu trenutno najboljeg rješenja
#                                     # epsilon = mali slučajni broj / slučajan mali pomak
#                                     # average_loudness = prosječna glasnoća svih šišmiša
#                                     # ovo služi za lokalnu pretragu oko najboljeg rješenja
 
#     x_new = popravi_granice(x_new)
#                                     # ako je nova pozicija izašla van dozvoljenog opsega,
#                                     # vrati je unutar granica problema
 
#     fitness_new = objective(x_new)
#                                     # izračunaj koliko je dobra nova pozicija
#                                     # objective = funkcija koju minimiziraš ili maksimiziraš
 
#     ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
#         x[i] = x_new
#                                     # prihvati novu poziciju za tog šišmiša
 
#         fitness[i] = fitness_new
#                                     # zapamti novu vrijednost funkcije za tog šišmiša
 
#         loudness[i] = alpha * loudness[i]
#                                     # smanji glasnoću tog šišmiša
#                                     # što iteracije više idu, šišmiš postaje "mirniji"

In [3]:
OPCIJE = {
    # tt_split, random_state, C, kernel, gamma
    # "tt_split": [0.1, 0.2, 0.22, 0.25, 0.29, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5],
    "tt_split": [0.2, 0.22, 0.25, 0.29],
    # "tt_split": np.arange(0.1, 0.5, 0.0001).tolist(),
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}


# "tt_split": np.arange(0.32, 0.34, 0.0001).tolist(),

# SVM - Support Vector Machine - algoritam za klasifikaciju i regresiju

# RandomSearchCV - metoda za pronalaženje najboljih hiperparametara modela
# GridSearchCV - metoda za pronalaženje najboljih hiperparametara modela - isprobava sve kombinacije


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import random
import numpy as np

In [5]:
# Učitavanje podataka
df = pd.read_csv("iris.csv") 

X = df.drop('species', axis=1)
y = df['species']

In [6]:
X.head()

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [7]:
y.head()

0    setosa
1    setosa
2    setosa
3    setosa
4    setosa
Name: species, dtype: str

In [8]:
# Ovaj dio koda radi evaluaciju eksperimenata dati u varijabli OPCIJE.
# Za svaki eksperiment, dijeli podatke na trening i test skup, trenira SVM model sa zadanim hiperparametrima, i računa F1 score na test skupu.
# Koristeci GridSearchCV ili RandomSearchCV bi bilo efikasnije, ali ovaj kod demonstrira osnovni pristup evaluacije.

# Najbolji parametri: 
best_params = {
    "tt_split": 0.29,
    "random_state": 0,
    "C": 0.1,
    "kernel": "linear",
    "gamma": "scale"
}

best_score = 0
best_model = None

for tt_split in OPCIJE["tt_split"]:
    for random_state in OPCIJE["random_state"]:
        for C in OPCIJE["C"]:
            for kernel in OPCIJE["kernel"]:
                for gamma in OPCIJE["gamma"]:

                    # Podjela podataka na trening i test skup
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tt_split, random_state=random_state)

                    # Treniranje SVM modela
                    model = SVC(C=C, kernel=kernel, gamma=gamma)
                    model.fit(X_train, y_train)

                    # Predviđanje na test skupu
                    y_pred = model.predict(X_test)

                    # Računanje F1 score
                    score = f1_score(y_test, y_pred, average='weighted')
                    # ovdje moze ici bilo koja funkcija EVALUACIJE, npr. accuracy_score, precision_score, recall_score, itd.

                    # Ispis rezultata za trenutne parametre
                    print(f"tt_split: {tt_split}, random_state: {random_state}, C: {C}, kernel: {kernel}, gamma: {gamma} => F1 Score: {score}")
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            "tt_split": tt_split,
                            "random_state": random_state,
                            "C": C,
                            "kernel": kernel,
                            "gamma": gamma
                        }
                        best_model = model
                    
                    
            

tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: scale => F1 Score: 0.8380018674136321
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: auto => F1 Score: 0.9672820512820512
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf,

In [9]:
best_params, best_score


({'tt_split': 0.2,
  'random_state': 0,
  'C': 0.1,
  'kernel': 'linear',
  'gamma': 'scale'},
 1.0)

In [10]:
best_model


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [11]:
# Random Search za trazenje boljih hiperparametara SVM-a
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC

# Napomena:
# tt_split nije parametar SVC modela pa ne moze ici u RandomizedSearchCV.
# Zato ovdje pretrazujemo samo stvarne hiperparametre koje SVC prima.

# Koristimo dva odvojena prostora pretrage:
# 1. linear kernel
# 2. rbf kernel
# Na taj nacin izbjegavamo invalidne kombinacije parametara.

PROSTOR_PRETRAGE = [
    {
        "kernel": ["linear"],
        "C": [0.1, 0.5, 1, 2, 10, 25, 50],
        "class_weight": [None, "balanced"],
        "shrinking": [True, False],
        "tol": [1e-3, 1e-4, 1e-5],
        "max_iter": [-1, 500, 1000],
        "decision_function_shape": ["ovr"],
        "break_ties": [True, False],
        "probability": [False]
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 0.5, 1, 2, 10, 25, 50],
        "gamma": ["scale", "auto"],
        "class_weight": [None, "balanced"],
        "shrinking": [True, False],
        "tol": [1e-3, 1e-4, 1e-5],
        "max_iter": [-1, 500, 1000],
        "decision_function_shape": ["ovr"],
        "break_ties": [True, False],
        "probability": [False]
    }
]

random_search = RandomizedSearchCV(
    estimator=SVC(),
    param_distributions=PROSTOR_PRETRAGE,
    n_iter=30,
    scoring="f1_weighted",
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

print("Najbolji parametri (Random Search):")
print(random_search.best_params_)
print(f"Najbolji F1 score: {random_search.best_score_:.4f}")

Najbolji parametri (Random Search):
{'tol': 1e-05, 'shrinking': False, 'probability': False, 'max_iter': -1, 'kernel': 'rbf', 'gamma': 'scale', 'decision_function_shape': 'ovr', 'class_weight': None, 'break_ties': True, 'C': 2}
Najbolji F1 score: 0.9799


In [12]:
# Bat Algorithm za optimizaciju SVM hiperparametara
# Svaki sismis predstavlja jedno kandidat-rjesenje:
# pozicija = [log10(C), log10(gamma)]
# kernel posebno biramo iz skupa ["linear", "rbf"]

from sklearn.model_selection import cross_val_score

BROJ_SISMISA = 12
BROJ_ITERACIJA = 25

F_MIN = 0.0
F_MAX = 2.0

ALPHA = 0.85          # smanjenje glasnoce
GAMMA_PULSE = 0.9     # rast pulse rate
POCETNA_GLASNOCA = 1.0
POCETNI_PULSE = 0.5

DONJA_GRANICA = np.array([-2.0, -3.0])   # C: 10^-2, gamma: 10^-3
GORNJA_GRANICA = np.array([ 2.0,  1.0])  # C: 10^2,  gamma: 10^1

MOGUCI_KERNELI = ["linear", "rbf"]


def evaluiraj_rjesenje(pozicija, kernel):
    C_vrijednost = 10 ** pozicija[0]
    gamma_vrijednost = 10 ** pozicija[1]

    if kernel == "linear":
        model = SVC(C=C_vrijednost, kernel=kernel)
    else:
        model = SVC(C=C_vrijednost, gamma=gamma_vrijednost, kernel=kernel)

    scores = cross_val_score(model, X, y, cv=5, scoring="f1_weighted")
    return scores.mean()


def vrati_u_granice(pozicija):
    return np.clip(pozicija, DONJA_GRANICA, GORNJA_GRANICA)


np.random.seed(42)
random.seed(42)

pozicije = np.random.uniform(DONJA_GRANICA, GORNJA_GRANICA, (BROJ_SISMISA, 2))
brzine = np.zeros((BROJ_SISMISA, 2))
glasnoce = np.full(BROJ_SISMISA, POCETNA_GLASNOCA)
pulse_rate = np.full(BROJ_SISMISA, POCETNI_PULSE)
kerneli = [random.choice(MOGUCI_KERNELI) for _ in range(BROJ_SISMISA)]

fitness = np.array([evaluiraj_rjesenje(pozicije[i], kerneli[i]) for i in range(BROJ_SISMISA)])

indeks_najboljeg = np.argmax(fitness)
najbolja_pozicija = pozicije[indeks_najboljeg].copy()
najbolji_kernel = kerneli[indeks_najboljeg]
najbolji_fitness = fitness[indeks_najboljeg]

print("Pocetno najbolje rjesenje:")
print(f"F1 = {najbolji_fitness:.4f}, C = {10**najbolja_pozicija[0]:.4f}, gamma = {10**najbolja_pozicija[1]:.4f}, kernel = {najbolji_kernel}")

for iteracija in range(1, BROJ_ITERACIJA + 1):
    prosjecna_glasnoca = np.mean(glasnoce)

    for i in range(BROJ_SISMISA):
        beta = random.random()
        frekvencija = F_MIN + (F_MAX - F_MIN) * beta

        brzine[i] = brzine[i] + (pozicije[i] - najbolja_pozicija) * frekvencija
        nova_pozicija = pozicije[i] + brzine[i]
        novi_kernel = kerneli[i]

        if random.random() > pulse_rate[i]:
            epsilon = np.random.uniform(-1, 1, 2)
            nova_pozicija = najbolja_pozicija + epsilon * prosjecna_glasnoca
            novi_kernel = random.choice(MOGUCI_KERNELI)

        nova_pozicija = vrati_u_granice(nova_pozicija)
        novi_fitness = evaluiraj_rjesenje(nova_pozicija, novi_kernel)

        if novi_fitness > fitness[i] and random.random() < glasnoce[i]:
            pozicije[i] = nova_pozicija
            kerneli[i] = novi_kernel
            fitness[i] = novi_fitness

            glasnoce[i] = ALPHA * glasnoce[i]
            pulse_rate[i] = POCETNI_PULSE * (1 - np.exp(-GAMMA_PULSE * iteracija))

        if fitness[i] > najbolji_fitness:
            najbolji_fitness = fitness[i]
            najbolja_pozicija = pozicije[i].copy()
            najbolji_kernel = kerneli[i]

    print(
        f"Iteracija {iteracija:2d}/{BROJ_ITERACIJA} | "
        f"Best F1 = {najbolji_fitness:.4f} | "
        f"C = {10**najbolja_pozicija[0]:.4f} | "
        f"gamma = {10**najbolja_pozicija[1]:.4f} | "
        f"kernel = {najbolji_kernel}"
    )

print("\nBat Algorithm - finalni rezultat")
print(f"Najbolji F1 score: {najbolji_fitness:.4f}")
print(f"Najbolji kernel: {najbolji_kernel}")
print(f"Najbolji C: {10**najbolja_pozicija[0]:.4f}")
print(f"Najbolji gamma: {10**najbolja_pozicija[1]:.4f}")

Pocetno najbolje rjesenje:
F1 = 0.9866, C = 0.5343, gamma = 0.0146, kernel = linear
Iteracija  1/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  2/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  3/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  4/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  5/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  6/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  7/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  8/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija  9/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija 10/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Iteracija 11/25 | Best F1 = 0.9866 | C = 0.5343 | gamma = 0.0146 | kernel = linear
Ite